参数列表：\
1.alpha\
输出结果：\
1.NipLSD_choir.rds

In [ ]:
library(CHOIR)
library(Seurat)
library(dplyr)
library(ggplot2)  
setwd("/data/")

In [ ]:
input_rds <- "NipLSD_all_v5.rds"
n_cores <-12
sctransform <- FALSE
split_by <- "orig.source"
output_dir <- "."
alpha <- 0.5

In [ ]:
seurat.obj <- readRDS(input_rds)
cells_to_keep <- seurat.obj@meta.data[[split_by]] == "NipLSD5"
seurat.subset <- subset(seurat.obj, cells = colnames(seurat.obj)[cells_to_keep])

# 主处理函数
process_seurat <- function(seurat.obj, prefix, sample_name = NULL,alpha) {
  # 生成前缀
  if (!is.null(sample_name)) {
    prefix <- paste0(prefix, "_", gsub("[^[:alnum:]]", "_", sample_name))
  }
  
  cat(paste("Processing:", ifelse(!is.null(sample_name), sample_name, "all samples"), "\n"))    
  
  # 检查细胞数量
  if (ncol(seurat.obj) < 10) {
    warning(paste("Too few cells (", ncol(seurat.obj), ") for meaningful analysis. Skipping..."))
    return(NULL)
  }
  
  # 构建层次聚类树
  cat("Building hierarchical tree...\n")
  seurat.obj <- tryCatch({
    if(sctransform) {
      use_assay <- "SCT"
      buildTree(seurat.obj, use_assay = use_assay, n_cores = min(2, n_cores),alpha = alpha)
    } else {
      use_assay <- "RNA"
      buildTree(seurat.obj, use_assay = use_assay, use_slot=SeuratObject::Layers(seurat.obj)[2], n_cores = min(2, n_cores),alpha = alpha)
    }
    
  }, error = function(e) {
    warning(paste("buildTree failed:", e$message))
    return(NULL)
  })
  
  if (is.null(seurat.obj)) {
    return(NULL)
  }
  
  # 修剪层次聚类树
  cat("Pruning tree...\n")
  seurat.obj <- tryCatch({
    pruneTree(seurat.obj, n_cores = min(2, n_cores), alpha = alpha)
  }, error = function(e) {
    warning(paste("pruneTree failed:", e$message))
    return(NULL)
  })
  
  if (is.null(seurat.obj)) {
    return(NULL)
  }
  
  # 运行UMAP降维
  cat("Running UMAP...\n")
  seurat.obj <- tryCatch({
    runCHOIRumap(seurat.obj, reduction = "P0_reduction")
  }, error = function(e) {
    warning(paste("runCHOIRumap failed:", e$message))
    return(NULL)
  })
  
  if (is.null(seurat.obj)) {
    return(NULL)
  }
  
  # 检查CHOIR_clusters_0.05是否存在
  cluster_col <- paste0("CHOIR_clusters_", alpha)
  if (!cluster_col %in% colnames(seurat.obj@meta.data)) {
    warning(paste(cluster_col, "not found in metadata. Skipping clustering visualization."))
    return(seurat.obj)
  }
  
  # 设置当前的聚类结果为 CHOIR_clusters_0.05
  Idents(seurat.obj) <- cluster_col
  
  # 可视化聚类结果
  pdf(file = file.path(output_dir, paste0(prefix, "_choir_clusters.pdf")), width = 8, height = 8)
  p <- tryCatch({
     plotCHOIR(seurat.obj)
  }, error = function(e) {
    warning(paste("DimPlot failed:", e$message))
    return(NULL)
  })
  if (!is.null(p)) print(p)
  dev.off()
   
  # 保存处理后的RDS
  save_path <- file.path(output_dir, paste0(prefix, "_after_choir.rds"))
  cat(paste("Saving processed Seurat object to:", save_path, "\n"))
  saveRDS(seurat.obj, file = save_path)
  
  return(seurat.obj)
}

# 额外

In [ ]:
#标准化处理+harmony
# 线性降维 (PCA)
seurat_obj <- NormalizeData(seurat_obj) %>% 
  FindVariableFeatures(nfeatures = 3000) %>% 
  ScaleData()
seurat_obj <- RunPCA(seurat_obj, verbose = FALSE)
# 使用Harmony进行批次校正（基于PCA空间）
seurat_obj <- RunHarmony(seurat_obj, 
                         group.by.vars = "orig.source",  
                         reduction = "pca", 
                         reduction.save = "harmony", 
                         assay.use = "RNA", 
                         project.dim = FALSE)
seurat_obj <- RunUMAP(seurat_obj, dims = 1:30, reduction = "harmony", reduction.name = "umapharmony")

In [ ]:
saveRDS(seurat_obj,"NipLSD_choir_harmony_0.5.rds")

In [ ]:
seurat_obj <- subset(seurat_obj, subset = orig.source == "NipLSD5")
plot_d5 <- DimPlot(seurat_obj, 
                    reduction = "umapharmony", 
                    group.by = "CHOIR_clusters_0.05",
                    pt.size = 0.1,
                    label = FALSE,
                    repel = TRUE) +  # 设置高分辨率
  ggtitle("D5_CHOIR结果检查") +
  theme_classic(base_size = 12) +
  theme(plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
        legend.position = "right",
        legend.box = "vertical",
        legend.box.just = "left",
        legend.key.size = unit(0.8, "cm"),
        legend.text = element_text(size = 10),
        legend.title = element_text(size = 12, face = "bold"),
        legend.spacing.y = unit(0.2, "cm"))
ggsave(filename = "D5_CHOIR结果检查.png", plot = plot_d5, width = 14, height = 7, dpi = 300, bg = "white")